In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import time
import copy

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)
import matplotlib.pyplot as plt

import mlflow
import mlflow.pytorch


from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset

from torch.utils.data import DataLoader, random_split



## Build a ResNet18 Model From Screatch

# ResNet18 Architecture: Input to Fully Connected Layer

| Step | Layer / Operation | Output Shape | Explanation |
|---|---|---|---|
| 1 | Input | `(32, 3, 224, 224)` | Batch of 32 RGB images, each 224 × 224 pixels |
| 2 | `Conv2d(3, 64, kernel_size=7, stride=2, padding=3)` | `(32, 64, 112, 112)` | Extracts initial features and reduces spatial dimensions |
| 3 | `BatchNorm2d(64)` | `(32, 64, 112, 112)` | Normalizes feature maps |
| 4 | `ReLU` | `(32, 64, 112, 112)` | Introduces non-linearity |
| 5 | `MaxPool2d(kernel_size=3, stride=2, padding=1)` | `(32, 64, 56, 56)` | Reduces spatial dimensions |
| 6 | `layer1` — 2 BasicBlocks | `(32, 64, 56, 56)` | Learns features while preserving spatial dimensions |
| 7 | `layer2` — 2 BasicBlocks | `(32, 128, 28, 28)` | Doubles channels and halves spatial dimensions |
| 8 | `layer3` — 2 BasicBlocks | `(32, 256, 14, 14)` | Learns more complex features |
| 9 | `layer4` — 2 BasicBlocks | `(32, 512, 7, 7)` | Learns high-level semantic features |
| 10 | `AdaptiveAvgPool2d((1, 1))` | `(32, 512, 1, 1)` | Converts each feature map into one value |
| 11 | `torch.flatten(x, 1)` | `(32, 512)` | Converts each image's features into a vector |
| 12 | `Linear(512, 10)` | `(32, 10)` | Produces 10 class logits for each image |

## Important Notes

- **Shape format:** `(Batch, Channels, Height, Width)`.
- **BasicBlock:** Contains two `3×3` convolutions, Batch Normalization, ReLU, and a residual shortcut.
- **Shortcut:** Uses `Identity` when dimensions match, or a projection with `1×1 Conv` when dimensions change.
- **Adaptive Average Pooling:** Converts each `7×7` feature map into a single value.
- **Flatten:** Converts the pooled tensor into a 512-dimensional vector per image.
- **Fully Connected Layer:** Produces 10 logits because the model is configured for CIFAR-10.

**Note:** This table assumes `224×224` input images. CIFAR-10 images are originally `32×32`; resizing them to 224×224 or modifying the stem changes the practical architecture choices.

In [12]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1):
        super().__init__()
        
        # First convolutional layer
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        
        # Second convolutional layer
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        if in_channels != out_channels or stride != 1:
            
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()
        self.relu = nn.ReLU(inplace=True)
        
        

    def forward(self, x):
        # The shourtcut connection is used to match the dimensions of the input and output if they are differnent
        identity =  self.shortcut(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
       
        # Adding the identity (skip connection or the shourtcut)
        out = out + identity
        out = self.relu(out)
        
        return out
            
        
    def _make_layer(self, out_channels, num_blocks, stride):
        layers = []

        # First block may change channels and spatial size
        layers.append(
            BasicBlock(
                self.in_channels,
                out_channels,
                stride=stride
            )
        )

        # Update channels for the next blocks
        self.in_channels = out_channels

        # Remaining blocks keep the same dimensions
        for _ in range(1, num_blocks):
            layers.append(
                BasicBlock(
                    self.in_channels,
                    out_channels,
                    stride=1
                )
            )

        return nn.Sequential(*layers)

            


In [13]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.in_channels = 64

        # Stem
        self.conv1 = nn.Conv2d(
            3, 64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(
            kernel_size=3,
            stride=2,
            padding=1
        )

        # Four residual layers
        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)

        # Classification head
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_channels, num_blocks, stride):
        layers = []

        layers.append(
            BasicBlock(
                self.in_channels,
                out_channels,
                stride=stride
            )
        )

        self.in_channels = out_channels

        for _ in range(1, num_blocks):
            layers.append(
                BasicBlock(
                    self.in_channels,
                    out_channels,
                    stride=1
                )
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        # Stem
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.maxpool(x)

        # Residual layers
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        # Classification head
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)

        return x


In [14]:

model = ResNet18(num_classes=10)

images = torch.randn(32, 3, 224, 224)

outputs = model(images)

print("Output shape:", outputs.shape)
print("Total parameters:", sum(p.numel() for p in model.parameters()))


Output shape: torch.Size([32, 10])
Total parameters: 11186378
